# 🗺️ Day 03a: Routing — Static, Dynamic, RIP, OSPF & BGP

---

## 🎯 What You'll Master Today
- Static vs dynamic routing
- Distance vector (RIP — Bellman-Ford)
- Link state (OSPF — Dijkstra)
- Path vector (BGP — Internet backbone)
- Routing table & longest prefix match

---

### 🎭 The Analogy

**Static routing** = GPS pre-programmed with fixed directions.  
**Dynamic routing** = GPS that learns from traffic updates and adjusts in real-time.  
- **RIP**: Ask neighbors "how far is X?" (distance vector)  
- **OSPF**: Get a full map of the entire city, compute best route (link state)  
- **BGP**: Inter-city signposts between countries (path vector)

```
╔══════════════════════════════════════════════════════════════════════╗
║                  ROUTING PROTOCOL LANDSCAPE                         ║
╠══════════════════════════════════════════════════════════════════════╣
║                                                                      ║
║  Interior Gateway Protocols (IGP) — within one AS:                  ║
║    ┌──────────────┐       ┌──────────────┐                          ║
║    │  RIP (DV)    │       │  OSPF (LS)   │                          ║
║    │  Bellman-Ford │       │  Dijkstra    │                          ║
║    │  Max 15 hops │       │  Area-based  │                          ║
║    │  Simple/small│       │  Enterprise  │                          ║
║    └──────────────┘       └──────────────┘                          ║
║                                                                      ║
║  Exterior Gateway Protocol (EGP) — between ASes:                    ║
║    ┌──────────────────────────────────────┐                          ║
║    │  BGP (Path Vector)                    │                          ║
║    │  "The routing protocol of the Internet" │                       ║
║    │  70,000+ ASes, 900K+ routes           │                          ║
║    └──────────────────────────────────────┘                          ║
║                                                                      ║
╚══════════════════════════════════════════════════════════════════════╝
```

---

In [ ]:
# ============================================
# 1. Routing Table & Longest Prefix Match
# ============================================

class RoutingTable:
    def __init__(self):
        self.routes = []  # (network_int, prefix_len, next_hop, interface)
    
    def _ip_to_int(self, ip):
        return sum(int(o) << (8 * (3 - i)) for i, o in enumerate(ip.split('.')))
    
    def add_route(self, network_cidr, next_hop, interface):
        ip, prefix = network_cidr.split('/')
        prefix = int(prefix)
        net_int = self._ip_to_int(ip)
        self.routes.append((net_int, prefix, next_hop, interface))
    
    def lookup(self, dest_ip):
        """Longest prefix match — most specific route wins."""
        dest_int = self._ip_to_int(dest_ip)
        best_match = None
        best_prefix = -1
        
        for net_int, prefix, next_hop, iface in self.routes:
            mask = (0xFFFFFFFF << (32 - prefix)) & 0xFFFFFFFF
            if (dest_int & mask) == (net_int & mask):
                if prefix > best_prefix:
                    best_prefix = prefix
                    best_match = (net_int, prefix, next_hop, iface)
        
        return best_match
    
    def show(self):
        print("  ═══ Routing Table ═══\n")
        print(f"  {'Destination':>18} {'Next Hop':>15} {'Interface':>12}")
        print(f"  {'─'*18} {'─'*15} {'─'*12}")
        for net_int, prefix, next_hop, iface in self.routes:
            ip = '.'.join(str((net_int >> (8*i)) & 0xFF) for i in range(3,-1,-1))
            print(f"  {ip + '/' + str(prefix):>18} {next_hop:>15} {iface:>12}")

rt = RoutingTable()
rt.add_route('0.0.0.0/0',        '10.0.0.1',     'eth0')   # Default
rt.add_route('192.168.1.0/24',   '192.168.1.1',  'eth1')
rt.add_route('192.168.1.128/25', '192.168.1.129', 'eth2')  # More specific
rt.add_route('10.0.0.0/8',       '10.0.0.1',     'eth0')

rt.show()

# Test lookups
print("\n  ═══ Longest Prefix Match Lookups ═══\n")
for dest in ['192.168.1.50', '192.168.1.200', '10.5.5.5', '8.8.8.8']:
    match = rt.lookup(dest)
    if match:
        net_int, prefix, nh, iface = match
        ip = '.'.join(str((net_int >> (8*i)) & 0xFF) for i in range(3,-1,-1))
        print(f"    {dest:>16} → matched /{prefix} → next hop {nh} via {iface}")
    else:
        print(f"    {dest:>16} → no match (drop)")

In [ ]:
# ============================================
# 2. Bellman-Ford (RIP Simulation)
# ============================================

def bellman_ford_rip(graph, source):
    """
    RIP uses Bellman-Ford: each router tells neighbors its distance table.
    Max 15 hops. 16 = infinity (unreachable).
    """
    INF = float('inf')
    dist = {node: INF for node in graph}
    next_hop = {node: None for node in graph}
    dist[source] = 0
    next_hop[source] = source
    
    nodes = list(graph.keys())
    edges = [(u, v, w) for u in graph for v, w in graph[u]]
    
    print(f"  ═══ Bellman-Ford (RIP) from {source} ═══\n")
    
    for iteration in range(len(nodes) - 1):
        updated = False
        for u, v, w in edges:
            if dist[u] + w < dist[v]:
                dist[v] = dist[u] + w
                next_hop[v] = next_hop[u] if next_hop[u] != source else v
                updated = True
        
        print(f"    Iteration {iteration + 1}: {dict((k, dist[k]) for k in sorted(dist))}")
        if not updated:
            print(f"    (Converged early!)")
            break
    
    print(f"\n  Distance table from {source}:")
    for node in sorted(dist):
        hop_count = dist[node]
        status = '✅' if hop_count <= 15 else '❌ unreachable (>15 hops)'
        print(f"    {node}: cost={hop_count} {status}")

# Network topology
network = {
    'A': [('B', 1), ('C', 4)],
    'B': [('A', 1), ('C', 2), ('D', 5)],
    'C': [('A', 4), ('B', 2), ('D', 1)],
    'D': [('B', 5), ('C', 1), ('E', 3)],
    'E': [('D', 3)],
}

bellman_ford_rip(network, 'A')

In [ ]:
# ============================================
# 3. Dijkstra (OSPF Simulation)
# ============================================
import heapq

def dijkstra_ospf(graph, source):
    """
    OSPF runs Dijkstra on the full link-state database.
    Each router has complete topology map → computes SPF tree.
    """
    dist = {node: float('inf') for node in graph}
    dist[source] = 0
    prev = {node: None for node in graph}
    visited = set()
    pq = [(0, source)]
    
    print(f"  ═══ Dijkstra (OSPF SPF) from {source} ═══\n")
    
    step = 0
    while pq:
        d, u = heapq.heappop(pq)
        if u in visited:
            continue
        visited.add(u)
        step += 1
        
        print(f"    Step {step}: Visit {u} (cost={d})")
        
        for v, w in graph[u]:
            if v not in visited and d + w < dist[v]:
                dist[v] = d + w
                prev[v] = u
                heapq.heappush(pq, (dist[v], v))
    
    # Build paths
    print(f"\n  Shortest Path Tree from {source}:")
    for node in sorted(dist):
        path = []
        current = node
        while current:
            path.append(current)
            current = prev[current]
        path.reverse()
        print(f"    {node}: cost={dist[node]:>2}  path={'→'.join(path)}")

dijkstra_ospf(network, 'A')

In [ ]:
# ============================================
# 4. Protocol Comparison Table
# ============================================

print("""
  ╔═══════════════════════════════════════════════════════════════════════╗
  ║                   ROUTING PROTOCOL COMPARISON                        ║
  ╠════════════╤════════════════╤════════════════╤═══════════════════════╣
  ║  Feature   │     RIP        │     OSPF       │       BGP             ║
  ╠════════════╪════════════════╪════════════════╪═══════════════════════╣
  ║ Type       │ Distance Vector│ Link State     │ Path Vector           ║
  ║ Algorithm  │ Bellman-Ford   │ Dijkstra       │ Best Path Selection   ║
  ║ Metric     │ Hop count      │ Cost (bandwidth│ AS path + policies    ║
  ║            │                │ based)         │                       ║
  ║ Scope      │ IGP (small)    │ IGP (large)    │ EGP (Internet)        ║
  ║ Max hops   │ 15             │ No limit       │ No limit              ║
  ║ Convergence│ Slow (30s)     │ Fast           │ Slow (minutes)        ║
  ║ Updates    │ Full table     │ LSA (changes)  │ Path updates          ║
  ║            │ every 30s      │ on change      │ on change             ║
  ║ Transport  │ UDP :520       │ IP protocol 89 │ TCP :179              ║
  ║ Scalability│ ❌ Small only  │ ✅ Enterprise  │ ✅ Internet-scale     ║
  ║ Loop prev. │ Split horizon, │ SPF tree       │ AS path check         ║
  ║            │ poison reverse │ (loop-free)    │ (detect own AS#)      ║
  ╚════════════╧════════════════╧════════════════╧═══════════════════════╝

  Key Terms:
    AS (Autonomous System) = network under single admin (ISP, company)
    IGP = Interior Gateway Protocol (within one AS)
    EGP = Exterior Gateway Protocol (between ASes)
    LSA = Link State Advertisement (OSPF update packet)
""")

## 📝 Interview Questions

| # | Question | Key Point |
|---|----------|------------|
| 1 | Distance vector vs link state? | DV: share distance table with neighbors (RIP). LS: flood full topology, compute locally (OSPF) |
| 2 | What is longest prefix match? | Router picks the most specific (longest prefix) matching route. /25 beats /24 beats /0 |
| 3 | Why is RIP limited? | Max 15 hops. Slow convergence. Count-to-infinity problem. Only for small networks |
| 4 | What is BGP? | Path vector protocol. Routes between ASes. Uses TCP:179. Basis of the Internet |
| 5 | OSPF areas? | Divide large networks into areas. Area 0 = backbone. Each area runs SPF independently |

---

## ┌──────────────────────────────────────────────────────────┐
## │              📌 KEY TAKEAWAYS                            │
## ├──────────────────────────────────────────────────────────┤
## │                                                          │
## │  • Longest prefix match → most specific route wins       │
## │  • RIP: simple, hop count, Bellman-Ford, max 15 hops     │
## │  • OSPF: Dijkstra, link-state, area-based, enterprise    │
## │  • BGP: path vector, inter-AS, TCP:179, Internet backbone│
## │  • Static routes for small/fixed; dynamic for large nets │
## │                                                          │
## └──────────────────────────────────────────────────────────┘

---

## ⏭️ Next Up: **NAT & IPv6** — Address translation, IPv6 format, transition mechanisms